<div>
 
  <b>Escuela Politécnica Nacional</b><br>
  <small>Facultad de Ingeniería de Sistemas</small><br>
  <small>Ingeniería en Ciencias de la Computación</small>
 
  <hr>
 
  <div style="display:flex; justify-content:space-between;">
    <div>
      Estudiante: <b>Mateo Cumbal</b><br>
      Fecha: <b>2026-07-08</b>
    </div>
    <div style="text-align:right;">
      Asignatura: <b>Recuperación de la Información</b><br>
      Paralelo: <b>GR1CC</b>
    </div>
  </div>
 
  <hr>
 
  <div style="text-align:center;">
    <h1><b>Ejercicio 11 — Web Scraping</b></h1>
  </div>
 
</div>

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [ ]:
# !pip install beautifulsoup4 -q

In [10]:
from bs4 import BeautifulSoup

file = "rotisserie-chicken.html"

# Cargar el HTML semilla
with open(file, "r", encoding="utf-8") as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, "html.parser")

# Confirmar el selector del título (meta og:title)
titulo = soup.find("meta", {"property": "og:title"})["content"]
print("Título:", titulo)

# Confirmar el selector de ingredientes
ingredientes_tags = soup.find_all(
    "li", class_="mm-recipes-structured-ingredients__list-item"
)
print(f"\nIngredientes encontrados: {len(ingredientes_tags)}")
for ing in ingredientes_tags:
    print("-", ing.get_text().strip())

Título: Rotisserie Chicken

Ingredientes encontrados: 6
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [11]:
def extraer_receta(soup, url=None):
    """
    Extrae los datos de una receta a partir de su soup ya parseado.
    Retorna un dict listo para convertirse en fila de DataFrame.
    """
    # Título
    titulo_tag = soup.find("meta", {"property": "og:title"})
    titulo = titulo_tag["content"] if titulo_tag else None

    # Descripción
    desc_tag = soup.find("meta", {"name": "description"})
    descripcion = desc_tag["content"] if desc_tag else None

    # Ingredientes
    ingredientes_tags = soup.find_all(
        "li", class_="mm-recipes-structured-ingredients__list-item"
    )
    ingredientes = [tag.get_text().strip() for tag in ingredientes_tags]

    # Instrucciones
    instrucciones_tags = soup.find_all(
        "p", class_="comp mntl-sc-block mntl-sc-block-html"
    )
    instrucciones = [tag.get_text().strip() for tag in instrucciones_tags]

    # Nutrición
    nutricion_tags = soup.find_all(
        "span",
        class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix",
    )
    nutricion = [
        tag.parent.get_text().strip().replace("\n", " ") for tag in nutricion_tags
    ]

    return {
        "url": url,
        "titulo": titulo,
        "descripcion": descripcion,
        "ingredientes": ingredientes,
        "instrucciones": instrucciones,
        "nutricion": nutricion,
    }


# Prueba sobre la semilla
receta_semilla = extraer_receta(soup, url="rotisserie-chicken.html")
receta_semilla

{'url': 'rotisserie-chicken.html',
 'titulo': 'Rotisserie Chicken',
 'descripcion': "Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.",
 'ingredientes': ['1 (3 pound) whole chicken',
  '1 pinch salt',
  '¼ cup butter, melted',
  '1 tablespoon salt',
  '1 tablespoon ground paprika',
  '¼ tablespoon ground black pepper'],
 'instrucciones': ["Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.",
  "Here's what you'll need to make rotisserie chicken at home:",
  "· Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, peppe

In [7]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [14]:
import re

PATRON_RECETA = re.compile(r"^https://www\.allrecipes\.com/recipe/\d+/[\w-]+/?$")


def extraer_links_receta(soup, url_propia=None):
    """
    Extrae los links a otras recetas individuales dentro del soup,
    siguiendo el patrón real de una receta: /recipe/<id>/<slug>/

    Args:
        soup       : BeautifulSoup ya parseado de una página de receta
        url_propia : URL de la receta actual, para excluirla de sus propios links

    Returns:
        list de URLs únicas de recetas válidas
    """
    links_validos = set()

    for tag in soup.find_all("a", href=True):
        href = tag["href"].split("?")[0]  # quitar query strings (utm_source, etc.)
        if PATRON_RECETA.match(href) and href != url_propia:
            links_validos.add(href)

    return sorted(links_validos)


links_receta = extraer_links_receta(soup)
print(f"Links de receta válidos encontrados: {len(links_receta)}\n")
for url in links_receta:
    print(url)

Links de receta válidos encontrados: 16

https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/214618/beer-can-chicken/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe

In [17]:
!pip install requests -q

In [18]:
import time
import requests
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; EPN-IR-taller/1.0)"}
DELAY = 1.0  # segundos entre peticiones, para no saturar el servidor


def descargar_soup(url):
    """
    Descarga una página remota y retorna su BeautifulSoup ya parseado.
    """
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


# Construcción del corpus: semilla (local) + 16 recetas (remotas)
recetas = [receta_semilla]

for i, url in enumerate(links_receta, start=1):
    print(f"[{i}/{len(links_receta)}] Descargando: {url}")
    try:
        soup_receta = descargar_soup(url)
        recetas.append(extraer_receta(soup_receta, url=url))
    except requests.RequestException as e:
        print(f"  -> Error al descargar {url}: {e}")
    time.sleep(DELAY)

df_recetas = pd.DataFrame(recetas)
print(f"\nCorpus final: {len(df_recetas)} recetas")
df_recetas.head()

[1/16] Descargando: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
[2/16] Descargando: https://www.allrecipes.com/recipe/19944/drunk-chicken/
[3/16] Descargando: https://www.allrecipes.com/recipe/214618/beer-can-chicken/
[4/16] Descargando: https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
[5/16] Descargando: https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
[6/16] Descargando: https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
[7/16] Descargando: https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
[8/16] Descargando: https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
[9/16] Descargando: https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
[10/16] Descargando: https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
[11/16] Descargando: https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
[12/16] Descargando: https://www.allrecipes.com/recipe/27

,url,titulo,descripcion,ingredientes,instrucciones,nutricion
0,rotisserie-chicken.html,Rotisserie Chicken,Rotisserie chicken that's easy to cook on a ga...,"[1 (3 pound) whole chicken, 1 pinch salt, ¼ cu...",[Intimidated by the idea of making a rotisseri...,"[Total Fat 25g, Saturated Fat 10g, Cholesterol..."
1,https://www.allrecipes.com/recipe/14531/beer-b...,Beer Butt Chicken,"This beer butter chicken recipe combines beer,...","[1 cup butter, divided, 2 tablespoons garlic s...",[Preheat an outdoor grill for low heat and lig...,"[Total Carbohydrate 3g, Dietary Fiber 1g, Tota..."
2,https://www.allrecipes.com/recipe/19944/drunk-...,Drunk Chicken,Drunken chicken is cooked over the grill with ...,"[1 (2 to 3 pound) whole chicken, 1 (12 fluid o...",[Rinse and dry chicken. Remove excess fat and ...,"[Total Carbohydrate 6g, Dietary Fiber 1g, Tota..."
3,https://www.allrecipes.com/recipe/214618/beer-...,Beer Can Chicken,This beer can chicken cooks a perfectly season...,"[⅓ cup brown sugar, 2 tablespoons chili powder...",[Preheat a charcoal grill for medium-high heat...,"[Total Carbohydrate 24g, Dietary Fiber 3g, Tot..."
4,https://www.allrecipes.com/recipe/214619/bbq-b...,Best Beer Can Chicken,"In this best beer can chicken recipe, chicken ...","[2 cups cherry wood chips, ½ cup dark brown su...",[Soak wood chips in water for at least 1 hour....,"[Total Carbohydrate 23g, Dietary Fiber 4g, Tot..."


## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [19]:
from sentence_transformers import SentenceTransformer, util


# Texto combinado por receta: título + descripción + ingredientes + instrucciones
def construir_texto_receta(fila):
    partes = [
        fila["titulo"] or "",
        fila["descripcion"] or "",
        " ".join(fila["ingredientes"]),
        " ".join(fila["instrucciones"]),
    ]
    return " ".join(partes)


df_recetas["texto"] = df_recetas.apply(construir_texto_receta, axis=1)

modelo = SentenceTransformer("all-MiniLM-L6-v2")
embeddings_corpus = modelo.encode(df_recetas["texto"].tolist(), convert_to_tensor=True)

print(f"Embeddings generados: {embeddings_corpus.shape}")

c:\Users\Acer123\venvs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2878.73it/s]


Embeddings generados: torch.Size([17, 384])


In [30]:
def buscar_recetas(query, top_k=3):
    """
    Busca las recetas más relevantes para un query dado usando similitud coseno.
    Retorna las filas completas del corpus (incluyendo ingredientes e instrucciones),
    más la columna de similitud.
    """
    embedding_query = modelo.encode(query, convert_to_tensor=True)
    similitudes = util.cos_sim(embedding_query, embeddings_corpus)[0]
    top_indices = similitudes.argsort(descending=True)[:top_k]

    resultados = df_recetas.iloc[top_indices.tolist()].copy()
    resultados["similitud"] = similitudes[top_indices].tolist()
    return resultados


query_prueba = "chicken with beer for the grill"
buscar_recetas(query_prueba)[
    ["titulo", "url", "similitud"]
]  # vista resumida solo para inspección

,titulo,url,similitud
3,Beer Can Chicken,https://www.allrecipes.com/recipe/214618/beer-...,0.690536
2,Drunk Chicken,https://www.allrecipes.com/recipe/19944/drunk-...,0.685737
1,Beer Butt Chicken,https://www.allrecipes.com/recipe/14531/beer-b...,0.653905


In [24]:
!pip install python-dotenv google-genai -q

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
API_KEY = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)


def consultar_gemini(prompt):
    """Envía un prompt a Gemini 2.5 Flash, imprime la respuesta y la retorna."""
    respuesta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    print("\n--- Respuesta de Gemini ---")
    print(respuesta.text)
    print("---------------------------\n")
    return respuesta

In [31]:
def construir_prompt_rag(query, resultados_retrieval):
    """
    Construye el prompt de RAG combinando el query con las recetas recuperadas.
    Incluye ingredientes e instrucciones completas para que el modelo
    tenga contexto suficiente para elaborar la respuesta.
    """
    bloques = []
    for _, fila in resultados_retrieval.iterrows():
        bloque = (
            f"Título: {fila['titulo']}\n"
            f"Descripción: {fila['descripcion']}\n"
            f"Ingredientes: {', '.join(fila['ingredientes'])}\n"
            f"Instrucciones: {' '.join(fila['instrucciones'][:5])}\n"
            f"URL: {fila['url']}"
        )
        bloques.append(bloque)

    contexto = "\n\n".join(bloques)

    prompt = (
        "Eres un asistente de cocina experto. Usa ÚNICAMENTE la siguiente información "
        "de recetas para responder la consulta del usuario. Sé específico: menciona "
        "ingredientes clave, pasos relevantes, y justifica por qué la receta recomendada "
        "encaja con lo que pide el usuario.\n\n"
        f"Recetas disponibles:\n{contexto}\n\n"
        f"Consulta: {query}\n\n"
        "Responde de forma clara y útil, con al menos 3-4 oraciones."
    )
    return prompt


# --- Ejemplo 1: RAG con query temática ---
query1 = "chicken with beer for the grill"
resultados1 = buscar_recetas(query1, top_k=3)
prompt1 = construir_prompt_rag(query1, resultados1)
_ = consultar_gemini(prompt1)


--- Respuesta de Gemini ---
¡Claro que sí! Tengo varias opciones excelentes para pollo con cerveza para la parrilla. Todas las siguientes recetas se ajustan perfectamente a tu consulta:

1.  La receta **"Beer Can Chicken"** es ideal. Utiliza ingredientes clave como azúcar moreno, chile en polvo, pimentón, mostaza seca, y ½ lata de cerveza para un pollo entero de 3 libras. Las instrucciones relevantes incluyen precalentar una parrilla de carbón a fuego medio-alto, frotar el pollo con una mezcla de condimentos y cocinarlo sobre calor indirecto con la lata de cerveza insertada por aproximadamente 1 hora y 15 minutos, lo cual lo convierte en un método directo para pollo a la parrilla con cerveza.

2.  También te recomiendo la receta **"Drunk Chicken"**. Para esta receta, necesitarás un pollo entero, 1 lata de cerveza, sazonador para aves, un poco de humo líquido y hojas de laurel. El proceso implica insertar las hojas de laurel bajo la piel del pollo, sazonarlo abundantemente, verter humo

In [32]:
# --- Ejemplo 2: RAG con ingredientes disponibles ---
query2 = "I have chicken, butter, and paprika at home. What can I cook?"
resultados2 = buscar_recetas(query2, top_k=3)
prompt2 = construir_prompt_rag(query2, resultados2)
_ = consultar_gemini(prompt2)


--- Respuesta de Gemini ---
¡Claro que sí! Con pollo, mantequilla y pimentón en casa, te recomiendo preparar el **Beer Butt Chicken**.

Esta receta es perfecta para lo que tienes, ya que específicamente usa **1 (4 libras) pollo entero**, **1 taza de mantequilla (dividida)**, y **2 cucharadas de pimentón (dividido)**. Los pasos relevantes incluyen derretir 1/2 taza de mantequilla y mezclarla con 1 cucharada de sal de ajo, 1 cucharada de pimentón, sal y pimienta. Luego, se coloca el pollo en posición vertical sobre una lata de cerveza (con el resto de la mantequilla, sal de ajo y pimentón en la lata) y se rocía con la mantequilla sazonada derretida antes de cocinarlo a fuego bajo en la parrilla. Esta receta se ajusta perfectamente a tus ingredientes disponibles, utilizando todos ellos para crear un pollo húmedo y sabroso.
---------------------------

